In [ ]:
NOISY_DIR = '/content/drive/MyDrive/Colab Notebooks/Image restoration/train/NoisyLR'
GT_DIR = '/content/drive/MyDrive/Colab Notebooks/Image restoration/train/GT'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DenseBlock(nn.Module):
    """Single densely-connected block (5 convs, growth-channel style)."""

    def __init__(self, in_ch: int, growth_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, growth_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(in_ch + growth_ch, growth_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(in_ch + 2 * growth_ch, growth_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(in_ch + 3 * growth_ch, growth_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(in_ch + 4 * growth_ch, in_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        # residual scaling for training stability
        return x + 0.2 * x5

In [ ]:
class RRDB(nn.Module):
    """Residual in Residual Dense Block: 3 stacked DenseBlocks + outer residual."""

    def __init__(self, in_ch: int, growth_ch: int = 32):
        super().__init__()
        self.db1 = DenseBlock(in_ch, growth_ch)
        self.db2 = DenseBlock(in_ch, growth_ch)
        self.db3 = DenseBlock(in_ch, growth_ch)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return x + 0.2 * out

In [ ]:
class RestorationNet(nn.Module):
    """
    Joint denoise + 2x super-resolution network.

    Input:  degraded grayscale image, shape (B, 1, H, W), values in [0, 1]
            (input may slightly exceed [0,1] due to speckle noise pushing
            values out of range -- the network is trained to handle this,
            no hard clamping is done on the input).
    Output: restored grayscale image, shape (B, 1, 2H, 2W), values in [0, 1]
            (clamped at output since the ground truth is always valid range).

    Args:
        in_ch: input channels (1 for grayscale)
        feat_ch: base feature channel width (keep this small for 4GB VRAM,
                  e.g. 32-48; raise to 64 only if you have more VRAM)
        num_blocks: number of RRDB blocks (6-10 is a good range for a
                    hackathon-scale model; more blocks = better quality
                    but slower inference and more VRAM)
        scale: upsampling factor, 2 (e.g. 128->256 or 256->512)
    """

    def __init__(self, in_ch: int = 1, feat_ch: int = 48, num_blocks: int = 8,
                 growth_ch: int = 32, scale: int = 2):
        super().__init__()
        self.scale = scale

        self.conv_first = nn.Conv2d(in_ch, feat_ch, 3, 1, 1)

        self.body = nn.Sequential(
            *[RRDB(feat_ch, growth_ch) for _ in range(num_blocks)]
        )
        self.conv_body = nn.Conv2d(feat_ch, feat_ch, 3, 1, 1)

        # Upsampling: pixel-shuffle based, learned (not bicubic)
        # For scale=2, one pixel-shuffle stage is enough.
        upsample_layers = []
        n_upsample_stages = 1 if scale == 2 else 2  # supports scale=4 too
        for _ in range(n_upsample_stages):
            upsample_layers += [
                nn.Conv2d(feat_ch, feat_ch * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(negative_slope=0.2, inplace=True),
            ]
        self.upsample = nn.Sequential(*upsample_layers)

        self.conv_hr = nn.Conv2d(feat_ch, feat_ch, 3, 1, 1)
        self.conv_last = nn.Conv2d(feat_ch, in_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat  # global residual within feature space

        feat = self.upsample(feat)
        feat = self.lrelu(self.conv_hr(feat))
        out = self.conv_last(feat)

        # Add a bicubic-upsampled skip connection of the input so the
        # network only has to learn the *residual* detail/noise removal,
        # not reconstruct low-frequency content from scratch. This speeds
        # up convergence significantly.
        base = F.interpolate(x, scale_factor=self.scale, mode="bicubic",
                              align_corners=False)
        out = out + base

        return torch.clamp(out, 0.0, 1.0)

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

**Loss**

In [ ]:
class CharbonnierLoss(nn.Module):
    """Smooth, differentiable-everywhere approximation of L1 loss."""

    def __init__(self, eps: float = 1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))

In [ ]:
def _gaussian_window(window_size: int, sigma: float, device, dtype):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g.unsqueeze(0) * g.unsqueeze(1)
    return window_2d.unsqueeze(0).unsqueeze(0)

In [ ]:
class SSIMLoss(nn.Module):
    """1 - SSIM, computed per-batch, single-channel images assumed."""

    def __init__(self, window_size: int = 11, sigma: float = 1.5):
        super().__init__()
        self.window_size = window_size
        self.sigma = sigma
        self.register_buffer("window", torch.zeros(1))  # placeholder, built lazily
        self._built = False

    def _build_window(self, device, dtype):
        self.window = _gaussian_window(self.window_size, self.sigma, device, dtype)
        self._built = True

    def forward(self, pred, target, data_range: float = 1.0):
        if not self._built or self.window.device != pred.device:
            self._build_window(pred.device, pred.dtype)

        C1 = (0.01 * data_range) ** 2
        C2 = (0.03 * data_range) ** 2
        pad = self.window_size // 2

        mu1 = F.conv2d(pred, self.window, padding=pad)
        mu2 = F.conv2d(target, self.window, padding=pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 * mu1, mu2 * mu2, mu1 * mu2

        sigma1_sq = F.conv2d(pred * pred, self.window, padding=pad) - mu1_sq
        sigma2_sq = F.conv2d(target * target, self.window, padding=pad) - mu2_sq
        sigma12 = F.conv2d(pred * target, self.window, padding=pad) - mu1_mu2

        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / (
            (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
        )
        return 1.0 - ssim_map.mean()

In [ ]:
class SobelGradientLoss(nn.Module):
    """L1 loss between Sobel edge maps of prediction and target."""

    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.register_buffer("kx", kx.view(1, 1, 3, 3))
        self.register_buffer("ky", ky.view(1, 1, 3, 3))

    def _grad_mag(self, img):
        gx = F.conv2d(img, self.kx, padding=1)
        gy = F.conv2d(img, self.ky, padding=1)
        return torch.sqrt(gx * gx + gy * gy + 1e-6)

    def forward(self, pred, target):
        return F.l1_loss(self._grad_mag(pred), self._grad_mag(target))


In [ ]:
class FFTLoss(nn.Module):
    def forward(self, pred, target):
        pf = torch.fft.rfft2(pred, norm="ortho")
        tf = torch.fft.rfft2(target, norm="ortho")
        return F.l1_loss(torch.abs(pf), torch.abs(tf))

In [ ]:
from pytorch_msssim import MS_SSIM

class CombinedRestorationLoss(nn.Module):
    def __init__(self, w_pixel=1.0, w_ssim=0.5, w_edge=0.1, w_fft=0.05):
        super().__init__()
        self.pixel_loss = CharbonnierLoss()
        self.ssim_loss = MS_SSIM(data_range=1.0, size_average=True, channel=1)
        self.edge_loss = SobelGradientLoss()
        self.fft_loss = FFTLoss()
        self.w_pixel, self.w_ssim, self.w_edge, self.w_fft = w_pixel, w_ssim, w_edge, w_fft

    def forward(self, pred, target):
        l_pixel = self.pixel_loss(pred, target)
        l_ssim = 1.0 - self.ssim_loss(pred, target)
        l_edge = self.edge_loss(pred, target)
        l_fft = self.fft_loss(pred, target)
        total = self.w_pixel*l_pixel + self.w_ssim*l_ssim + self.w_edge*l_edge + self.w_fft*l_fft
        return total, {"pixel": l_pixel.item(), "ssim": l_ssim.item(),
                        "edge": l_edge.item(), "fft": l_fft.item(), "total": total.item()}

In [ ]:
def add_speckle_noise(img: torch.Tensor, noise_level: float) -> torch.Tensor:
    """Multiplicative speckle noise: out = img + img * N(0, noise_level^2)."""
    noise = torch.randn_like(img) * noise_level
    return img + img * noise


def add_gaussian_noise(img: torch.Tensor, noise_level: float) -> torch.Tensor:
    """Additive Gaussian sensor noise."""
    return img + torch.randn_like(img) * noise_level


def gaussian_blur(img: torch.Tensor, kernel_size: int = 5, sigma: float = 1.0) -> torch.Tensor:
    """Simple depthwise Gaussian blur, single-channel input (B,1,H,W)."""
    coords = torch.arange(kernel_size, dtype=img.dtype, device=img.device) - kernel_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).view(1, 1, -1)
    kernel_1d_h = g.view(1, 1, 1, kernel_size)
    kernel_1d_v = g.view(1, 1, kernel_size, 1)
    pad = kernel_size // 2
    img = F.conv2d(img, kernel_1d_h, padding=(0, pad))
    img = F.conv2d(img, kernel_1d_v, padding=(pad, 0))
    return img

In [ ]:
def degrade_image(
    gt: torch.Tensor,
    scale: int = 2,
    speckle_level_range=(0.05, 0.25),
    gaussian_level_range=(0.0, 0.03),
    blur_prob: float = 0.5,
    blur_sigma_range=(0.5, 1.2),
) -> torch.Tensor:
    """
    Args:
        gt: ground truth image tensor, shape (B, 1, H, W), values in [0, 1]
        scale: downsampling factor (2 -> half resolution)
    Returns:
        degraded image tensor, shape (B, 1, H/scale, W/scale).
        NOT clamped -- may exceed [0, 1], matching real degraded data.
    """
    x = gt

    if torch.rand(1).item() < blur_prob:
        sigma = float(np.random.uniform(*blur_sigma_range))
        x = gaussian_blur(x, kernel_size=5, sigma=sigma)

    # downsample
    h, w = x.shape[-2:]
    x = F.interpolate(x, size=(h // scale, w // scale), mode="bicubic",
                       align_corners=False)

    speckle_level = float(np.random.uniform(*speckle_level_range))
    x = add_speckle_noise(x, speckle_level)

    gauss_level = float(np.random.uniform(*gaussian_level_range))
    x = add_gaussian_noise(x, gauss_level)

    return x

In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import copy
from torch.utils.data import Dataset
from PIL import Image

In [ ]:
NPY_EXTENSIONS = {".npy"}

In [ ]:
def _list_npy(folder: str):
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in NPY_EXTENSIONS])

In [ ]:
def _load_grayscale(path) -> np.ndarray:
    img = Image.open(path).convert("L")  # force single-channel grayscale
    return np.asarray(img, dtype=np.float32) / 255.0

In [ ]:
def _load_npy(path) -> np.ndarray:
    """Load a single-image .npy file as a 2D (H, W) float32 array.

    Squeezes any singleton channel dim (e.g. (128,128,1) or (1,128,128))
    so downstream code can always assume a plain 2D array.
    """
    arr = np.load(path).astype(np.float32)
    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array after squeeze, got shape {arr.shape} for {path}")
    return arr

In [ ]:
def _random_crop_pair(degraded: np.ndarray, gt: np.ndarray, lr_patch: int, scale: int):
    """Crop matching patches: lr_patch from degraded, lr_patch*scale from gt."""
    h, w = degraded.shape
    if h < lr_patch or w < lr_patch:
        raise ValueError(
            f"Image too small ({h}x{w}) for requested patch size {lr_patch}."
        )
    top = random.randint(0, h - lr_patch)
    left = random.randint(0, w - lr_patch)
    d_patch = degraded[top:top + lr_patch, left:left + lr_patch]
    gt_top, gt_left, gt_size = top * scale, left * scale, lr_patch * scale
    g_patch = gt[gt_top:gt_top + gt_size, gt_left:gt_left + gt_size]
    return d_patch, g_patch


def _augment_pair(d: np.ndarray, g: np.ndarray):
    """Matched random flip/rotation augmentation applied to both images."""
    if random.random() < 0.5:
        d, g = np.fliplr(d).copy(), np.fliplr(g).copy()
    if random.random() < 0.5:
        d, g = np.flipud(d).copy(), np.flipud(g).copy()
    k = random.choice([0, 1, 2, 3])
    if k:
        d, g = np.rot90(d, k).copy(), np.rot90(g, k).copy()
    return d, g

In [ ]:
def tta_predict(model, x):
    outs = []
    for hflip in (False, True):
        for k in range(4):
            xi = torch.rot90(x, k, dims=(-2, -1))
            if hflip: xi = torch.flip(xi, dims=(-1,))
            with torch.no_grad():
                yi = model(xi)
            if hflip: yi = torch.flip(yi, dims=(-1,))
            yi = torch.rot90(yi, -k, dims=(-2, -1))
            outs.append(yi)
    return torch.stack(outs, 0).mean(0).clamp(0, 1)

In [ ]:
class PairedRestorationDataset(Dataset):
    """
    Real paired KLA data: NoisyLR/xxx.npy (128x128) <-> GT/xxx.npy (256x256).

    Since inputs are already exactly 128x128 and GT already exactly 256x256
    (fixed sizes, not variable-size images), by default this dataset uses
    the FULL image each time rather than random sub-patch cropping -- set
    lr_patch=None (the default) to use full 128x128 -> 256x256 pairs, which
    matches how the model will actually be evaluated (full images, not
    patches). Pass an explicit lr_patch (e.g. 64) if you want random-crop
    patch training instead (useful mainly for reducing VRAM/batch size).
    """

    def __init__(self, degraded_dir: str, gt_dir: str, lr_patch=None,
                 scale: int = 2, augment: bool = True):
        self.degraded_paths = _list_npy(degraded_dir)
        if not self.degraded_paths:
            raise FileNotFoundError(f"No .npy files found in {degraded_dir}")

        gt_dir = Path(gt_dir)
        self.gt_paths = [gt_dir / p.name for p in self.degraded_paths]
        missing = [gp for gp in self.gt_paths if not gp.exists()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} NoisyLR files have no matching GT file, "
                f"e.g. {missing[0]}. Check that filenames match exactly "
                f"between NoisyLR/ and GT/ folders."
            )

        self.lr_patch = lr_patch
        self.scale = scale
        self.augment = augment

    def __len__(self):
        return len(self.degraded_paths)

    def __getitem__(self, idx):
        degraded = _load_npy(self.degraded_paths[idx])
        gt = _load_npy(self.gt_paths[idx])

        expected_gt_shape = (degraded.shape[0] * self.scale, degraded.shape[1] * self.scale)
        if gt.shape != expected_gt_shape:
            raise ValueError(
                f"Shape mismatch for {self.degraded_paths[idx].name}: "
                f"degraded {degraded.shape} implies GT should be "
                f"{expected_gt_shape}, but GT is {gt.shape}."
            )

        if self.lr_patch is not None:
            d_patch, g_patch = _random_crop_pair(degraded, gt, self.lr_patch, self.scale)
        else:
            d_patch, g_patch = degraded, gt

        if self.augment:
            d_patch, g_patch = _augment_pair(d_patch, g_patch)

        d_t = torch.from_numpy(d_patch.copy()).unsqueeze(0).float()
        g_t = torch.from_numpy(g_patch.copy()).unsqueeze(0).float()
        return d_t, g_t

In [ ]:
class SyntheticRestorationDataset(Dataset):
    """Clean images only -> degradation generated on-the-fly."""

    def __init__(self, clean_dir: str, lr_patch: int = 64, scale: int = 2,
                 augment: bool = True):
        self.clean_paths = _list_images(clean_dir)
        if len(self.clean_paths) == 0:
            raise FileNotFoundError(f"No images found in {clean_dir}")
        self.lr_patch = lr_patch
        self.scale = scale
        self.augment = augment

    def __len__(self):
        return len(self.clean_paths)

    def __getitem__(self, idx):
        gt_full = _load_grayscale(self.clean_paths[idx])
        h, w = gt_full.shape
        gt_patch_size = self.lr_patch * self.scale
        if h < gt_patch_size or w < gt_patch_size:
            # pad small images up rather than skipping them
            pad_h = max(0, gt_patch_size - h)
            pad_w = max(0, gt_patch_size - w)
            gt_full = np.pad(gt_full, ((0, pad_h), (0, pad_w)), mode="reflect")
            h, w = gt_full.shape

        top = random.randint(0, h - gt_patch_size)
        left = random.randint(0, w - gt_patch_size)
        gt_patch = gt_full[top:top + gt_patch_size, left:left + gt_patch_size]

        if self.augment:
            k = random.choice([0, 1, 2, 3])
            if k:
                gt_patch = np.rot90(gt_patch, k).copy()
            if random.random() < 0.5:
                gt_patch = np.fliplr(gt_patch).copy()

        gt_t = torch.from_numpy(gt_patch).unsqueeze(0).unsqueeze(0).float()  # (1,1,H,W)
        degraded_t = degrade_image(gt_t, scale=self.scale).squeeze(0)  # (1,h,w)
        gt_t = gt_t.squeeze(0)  # (1,H,W)

        return degraded_t, gt_t


In [ ]:
import argparse
import sys
import time
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data import random_split
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

In [ ]:
def build_dataset(args):
    train_full = PairedRestorationDataset(args.degraded_dir, args.gt_dir,
                                           lr_patch=args.lr_patch, scale=args.scale, augment=True)
    val_full = PairedRestorationDataset(args.degraded_dir, args.gt_dir,
                                         lr_patch=args.lr_patch, scale=args.scale, augment=False)
    n_val = max(1, int(0.1 * len(train_full)))
    n_train = len(train_full) - n_val
    g = torch.Generator().manual_seed(42)
    perm = torch.randperm(len(train_full), generator=g).tolist()
    train_idx, val_idx = perm[:n_train], perm[n_train:]
    train_set = torch.utils.data.Subset(train_full, train_idx)
    val_set = torch.utils.data.Subset(val_full, val_idx)
    return train_set, val_set

In [ ]:
def icnr_init(conv: nn.Conv2d, scale: int = 2):
    """Initialize conv weights so PixelShuffle starts equivalent to nearest-neighbor
    upsampling — kills checkerboard/ripple artifacts from random init."""
    out_ch, in_ch, kh, kw = conv.weight.shape
    sub_ch = out_ch // (scale ** 2)
    k = torch.empty(sub_ch, in_ch, kh, kw)
    nn.init.kaiming_normal_(k)
    k = k.repeat_interleave(scale ** 2, dim=0)
    conv.weight.data.copy_(k)
    if conv.bias is not None:
        conv.bias.data.zero_()

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.clone()
    def apply_to(self, model):
        model.load_state_dict(self.shadow, strict=True)

In [ ]:
def main(argv=None):
    parser = argparse.ArgumentParser(description="Train restoration model")
    parser.add_argument("--degraded_dir", type=str, default=None)
    parser.add_argument("--gt_dir", type=str, default=None)
    parser.add_argument("--clean_dir", type=str, default=None,
                         help="Folder of clean images for synthetic degradation")
    parser.add_argument("--out_dir", type=str, default="checkpoints")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--lr_patch", type=int, default=None,
                         help="Degraded (LR) patch size for random-crop training. "
                              "Leave unset (None) to train on full 128x128->256x256 "
                              "pairs (matches real NoisyLR/GT data as-is). Set to e.g. "
                              "64 to train on random 64x64->128x128 crops instead "
                              "(reduces VRAM/batch size needs).")
    parser.add_argument("--scale", type=int, default=2)
    parser.add_argument("--feat_ch", type=int, default=48,
                         help="Model width; lower (e.g. 32) if VRAM-limited")
    parser.add_argument("--num_blocks", type=int, default=8,
                         help="Number of RRDB blocks; lower (e.g. 5-6) if VRAM-limited")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--save_every", type=int, default=1,
                         help="Save a checkpoint every N epochs. Default 1 -- "
                              "save every epoch, since Colab sessions can "
                              "disconnect at any time and you don't want to "
                              "lose progress.")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--resume", type=str, default=None,
                         help="Path to a checkpoint .pt to resume training from "
                              "(e.g. after a Colab disconnect). Restores model "
                              "weights and continues from the saved epoch number.")
    args = parser.parse_args(argv)

    os.makedirs(args.out_dir, exist_ok=True)
    print(f"Using device: {args.device}")

    train_set, val_set = build_dataset(args)
    loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,
                        num_workers=args.num_workers, pin_memory=(args.device=="cuda"), drop_last=True)
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=args.num_workers)

    model = RestorationNet(
        in_ch=1, feat_ch=args.feat_ch, num_blocks=args.num_blocks, scale=args.scale,
    ).to(args.device)
    print(f"Model parameters: {count_parameters(model):,}")

    for m in model.upsample:
      if isinstance(m, nn.Conv2d):
          icnr_init(m, scale=2)

    ema = EMA(model)

    criterion = CombinedRestorationLoss().to(args.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.99))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    start_epoch = 1
    best_loss = float("inf")
    best_val_ssim = -1.0

    if args.resume:
        if not Path(args.resume).exists():
            raise FileNotFoundError(f"--resume checkpoint not found: {args.resume}")
        print(f"Resuming from checkpoint: {args.resume}")
        ckpt = torch.load(args.resume, map_location=args.device)
        model.load_state_dict(ckpt["model_state_dict"])
        if "ema_state_dict" in ckpt:
            ema.shadow = ckpt["ema_state_dict"]
            print("Restored EMA shadow weights.")
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt.get("epoch", 0) + 1
        best_loss = ckpt.get("best_loss", float("inf"))
        best_val_ssim = ckpt.get("best_val_ssim", -1.0)
        print(f"Resuming from epoch {start_epoch} (best_loss so far: {best_loss:.4f})")

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        epoch_start = time.time()
        running = {"pixel": 0.0, "ssim": 0.0, "edge": 0.0, "fft": 0.0, "total": 0.0}

        for i, (degraded, gt) in enumerate(loader):
            degraded, gt = degraded.to(args.device), gt.to(args.device)

            optimizer.zero_grad()
            pred = model(degraded)
            loss, parts = criterion(pred, gt)
            loss.backward()
            optimizer.step()
            ema.update(model)

            for k in running:
                running[k] += parts[k]

            if (i + 1) % 20 == 0:
                n = i + 1
                print(f"  epoch {epoch} step {n}/{len(loader)} "
                      f"total={running['total']/n:.4f} "
                      f"pixel={running['pixel']/n:.4f} "
                      f"ssim={running['ssim']/n:.4f} "
                      f"edge={running['edge']/n:.4f}")

        scheduler.step()
        model.eval()

        eval_model = copy.deepcopy(model)
        ema.apply_to(eval_model)
        eval_model.eval()

        val_ssim, val_psnr = [], []
        with torch.no_grad():
            for degraded, gt in val_loader:
                degraded, gt = degraded.to(args.device), gt.to(args.device)
                pred = eval_model(degraded)
                p = pred.squeeze().cpu().numpy()
                g = gt.squeeze().cpu().numpy()
                val_ssim.append(ssim_fn(g, p, data_range=1.0))
                val_psnr.append(psnr_fn(g, p, data_range=1.0))
        val_ssim, val_psnr = float(np.mean(val_ssim)), float(np.mean(val_psnr))
        print(f"  [val] SSIM={val_ssim:.4f} PSNR={val_psnr:.2f}dB")
        model.train()
        n_batches = len(loader)
        avg_loss = running["total"] / max(n_batches, 1)
        elapsed = time.time() - epoch_start
        print(f"Epoch {epoch}/{args.epochs} done in {elapsed:.1f}s -- "
              f"avg_total_loss={avg_loss:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if epoch % args.save_every == 0:
            latest_path = Path(args.out_dir) / "model_latest.pt"
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "ema_state_dict": ema.shadow,          # ADD
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_loss": best_loss,
                "best_val_ssim": best_val_ssim,        # ADD
                "args": vars(args),
            }, latest_path)
            print(f"Saved checkpoint: {latest_path} (epoch {epoch})")

        if val_ssim > best_val_ssim:
            best_val_ssim = val_ssim
            best_path = Path(args.out_dir) / "model_best.pt"
            torch.save({
                "epoch": epoch,
                "model_state_dict": eval_model.state_dict(),
                "ema_state_dict": ema.shadow,
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_loss": best_loss,
                "best_val_ssim": best_val_ssim,
                "args": vars(args),
            }, best_path)

    ema.apply_to(model)
    final_path = Path(args.out_dir) / "model_final.pt"
    torch.save({
        "epoch": args.epochs,
        "model_state_dict": model.state_dict(),
        "args": vars(args),
    }, final_path)
    print(f"Training complete. Final model saved to {final_path}")

In [ ]:
main([
    "--degraded_dir", NOISY_DIR,
    "--gt_dir", GT_DIR,
    "--epochs", "20",
    "--batch_size", "8",
    "--out_dir", "/content/drive/MyDrive/Colab Notebooks/Image restoration/checkpoints/v3",
    "--resume", "/content/drive/MyDrive/Colab Notebooks/Image restoration/checkpoints/v3/model_latest.pt",
])

Using device: cuda
Model parameters: 4,608,817
  epoch 1 step 20/360 total=0.1607 pixel=0.0631 ssim=0.1514 edge=0.2045
  epoch 1 step 40/360 total=0.1454 pixel=0.0568 ssim=0.1381 edge=0.1817
  epoch 1 step 60/360 total=0.1354 pixel=0.0512 ssim=0.1311 edge=0.1727
  epoch 1 step 80/360 total=0.1305 pixel=0.0488 ssim=0.1263 edge=0.1720
  epoch 1 step 100/360 total=0.1260 pixel=0.0466 ssim=0.1224 edge=0.1690
  epoch 1 step 120/360 total=0.1230 pixel=0.0453 ssim=0.1192 edge=0.1670
  epoch 1 step 140/360 total=0.1217 pixel=0.0449 ssim=0.1175 edge=0.1678
  epoch 1 step 160/360 total=0.1202 pixel=0.0444 ssim=0.1155 edge=0.1677
  epoch 1 step 180/360 total=0.1176 pixel=0.0435 ssim=0.1126 edge=0.1648
  epoch 1 step 200/360 total=0.1155 pixel=0.0426 ssim=0.1106 edge=0.1628
  epoch 1 step 220/360 total=0.1148 pixel=0.0424 ssim=0.1097 edge=0.1628
  epoch 1 step 240/360 total=0.1136 pixel=0.0420 ssim=0.1082 edge=0.1620
  epoch 1 step 260/360 total=0.1124 pixel=0.0416 ssim=0.1068 edge=0.1612
  epoch 